In [3]:
import pandas as pd
import numpy as np
import sqlite3

In [4]:
conn = sqlite3.connect("nifty100.db")

In [5]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,companies
1,profitandloss
2,balancesheet
3,cashflow
4,market_cap


### Loading data from provided HTML files and inserting into SQLite database

In [6]:
companies = pd.read_csv('/content/companies_cleaned.csv')
profit = pd.read_csv('/content/profitandloss_cleaned.csv')
balance = pd.read_csv('/content/balancesheet_cleaned.csv')
cashflow = pd.read_csv('/content/cashflow_cleaned.csv')
market = pd.read_csv('/content/market_cap_cleaned.csv')

In [9]:
# Insert DataFrames into SQLite
companies.to_sql('companies', conn, if_exists='replace', index=False)
profit.to_sql('profitandloss', conn, if_exists='replace', index=False)
balance.to_sql('balancesheet', conn, if_exists='replace', index=False)
cashflow.to_sql('cashflow', conn, if_exists='replace', index=False)
market.to_sql('market_cap', conn, if_exists='replace', index=False)

print("Tables created in nifty100.db from HTML files.")

# Verify that tables now exist
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

Tables created in nifty100.db from HTML files.


,name
0,companies
1,profitandloss
2,balancesheet
3,cashflow
4,market_cap


In [10]:
companies = pd.read_sql("SELECT * FROM companies", conn)
profit = pd.read_sql("SELECT * FROM profitandloss", conn)
balance = pd.read_sql("SELECT * FROM balancesheet", conn)
cashflow = pd.read_sql("SELECT * FROM cashflow", conn)
market = pd.read_sql("SELECT * FROM market_cap", conn)

The tables have been loaded into the SQLite database. Now the original data loading cell should work.

In [11]:
companies.head()
profit.head()
balance.head()
cashflow.head()

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0


In [12]:
print(profit.columns)
print(balance.columns)
print(cashflow.columns)

Index(['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout'],
      dtype='object')
Index(['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings',
       'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip',
       'investments', 'other_asset', 'total_assets'],
      dtype='object')
Index(['id', 'company_id', 'year', 'operating_activity', 'investing_activity',
       'financing_activity', 'net_cash_flow'],
      dtype='object')


In [13]:
import re
import numpy as np
import pandas as pd


def safe_divide(numerator, denominator):
    """
    Safely divide two columns.
    If denominator is 0 or missing, return NaN instead of crashing.
    """
    return np.where(
        (denominator == 0) | pd.isna(denominator),
        np.nan,
        numerator / denominator
    )


def extract_year(year_value):
    """
    Converts values like 'Mar-23' into 2023.
    """
    year_str = str(year_value)
    match = re.search(r"(\d{2,4})", year_str)

    if not match:
        return np.nan

    year = int(match.group(1))

    if year < 100:
        return 2000 + year

    return year


In [19]:
key_cols = ["company_id", "year", "fiscal_year"]

print("Before removing duplicates:")
print("profit:", profit.duplicated(key_cols).sum())
print("balance:", balance.duplicated(key_cols).sum())
print("cashflow:", cashflow.duplicated(key_cols).sum())

profit = profit.drop_duplicates(subset=key_cols, keep="first")
balance = balance.drop_duplicates(subset=key_cols, keep="first")
cashflow = cashflow.drop_duplicates(subset=key_cols, keep="first")

print("After removing duplicates:")
print("profit:", profit.duplicated(key_cols).sum())
print("balance:", balance.duplicated(key_cols).sum())
print("cashflow:", cashflow.duplicated(key_cols).sum())

Before removing duplicates:
profit: 13
balance: 87
cashflow: 23
After removing duplicates:
profit: 0
balance: 0
cashflow: 0


In [20]:
profit["fiscal_year"] = profit["year"].apply(extract_year)
balance["fiscal_year"] = balance["year"].apply(extract_year)
cashflow["fiscal_year"] = cashflow["year"].apply(extract_year)

# Merge main financial statements
df = (
    profit
    .merge(balance, on=["company_id", "year", "fiscal_year"], how="inner", suffixes=("_pl", "_bs"))
    .merge(cashflow, on=["company_id", "year", "fiscal_year"], how="left")
)

# Merge company data for face value
df = df.merge(
    companies[["id", "face_value"]],
    left_on="company_id",
    right_on="id",
    how="left"
)

# Merge market cap / valuation data
df = df.merge(
    market,
    left_on=["company_id", "fiscal_year"],
    right_on=["company_id", "year"],
    how="left",
    suffixes=("", "_market")
)

df.head()



/tmp/ipykernel_8579/3927418518.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  profit["fiscal_year"] = profit["year"].apply(extract_year)
/tmp/ipykernel_8579/3927418518.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  balance["fiscal_year"] = balance["year"].apply(extract_year)
/tmp/ipykernel_8579/3927418518.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: ht

,id_pl,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,id_y,face_value,id,year_market,market_cap_crore,enterprise_value_crore,pe_ratio,pb_ratio,ev_ebitda,dividend_yield_pct
0,61,ABB,Dec 2012,1653,1451,202.0,12.0,33,0,19,...,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,62,ABB,Mar 2014,2276,2009,267.0,12.0,49,0,22,...,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,63,ABB,Mar 2015,2289,1977,312.0,14.0,48,0,15,...,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,64,ABB,Mar 2016,2614,2250,365.0,14.0,50,3,14,...,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,65,ABB,Mar 2017,2903,2505,398.0,14.0,57,2,16,...,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
ratios = pd.DataFrame()

ratios["company_id"] = df["company_id"]
ratios["year"] = df["year"]
ratios["fiscal_year"] = df["fiscal_year"]

equity = df["equity_capital"] + df["reserves"]
capital_employed = equity + df["borrowings"]
ebit = df["operating_profit"] - df["depreciation"]
ebitda = df["operating_profit"]

# -----------------------------
# 1. Profitability KPIs
# -----------------------------
ratios["net_profit_margin_pct"] = safe_divide(df["net_profit"], df["sales"]) * 100
ratios["operating_profit_margin_pct"] = safe_divide(df["operating_profit"], df["sales"]) * 100
ratios["pbt_margin_pct"] = safe_divide(df["profit_before_tax"], df["sales"]) * 100
ratios["ebit_margin_pct"] = safe_divide(ebit, df["sales"]) * 100
ratios["expense_ratio_pct"] = safe_divide(df["expenses"], df["sales"]) * 100
ratios["return_on_equity_pct"] = safe_divide(df["net_profit"], equity) * 100
ratios["return_on_capital_employed_pct"] = safe_divide(ebit, capital_employed) * 100
ratios["return_on_assets_pct"] = safe_divide(df["net_profit"], df["total_assets"]) * 100

# -----------------------------
# 2. Debt / Leverage KPIs
# -----------------------------
ratios["debt_to_equity"] = safe_divide(df["borrowings"], equity)
ratios["debt_to_assets"] = safe_divide(df["borrowings"], df["total_assets"])
ratios["debt_to_capital"] = safe_divide(df["borrowings"], capital_employed)
ratios["liabilities_to_assets"] = safe_divide(df["total_liabilities"], df["total_assets"])
ratios["interest_coverage"] = safe_divide(
    df["operating_profit"] + df["other_income"],
    df["interest"]
)
ratios["net_debt_cr"] = df["borrowings"] - df["investments"]
ratios["net_debt_to_ebitda"] = safe_divide(ratios["net_debt_cr"], ebitda)

# -----------------------------
# 3. Cash Flow KPIs
# -----------------------------
ratios["cash_from_operations_cr"] = df["operating_activity"]
ratios["free_cash_flow_cr"] = df["operating_activity"] + df["investing_activity"]
ratios["capex_cr"] = df["investing_activity"].abs()
ratios["cfo_margin_pct"] = safe_divide(df["operating_activity"], df["sales"]) * 100
ratios["fcf_margin_pct"] = safe_divide(ratios["free_cash_flow_cr"], df["sales"]) * 100
ratios["cfo_to_pat"] = safe_divide(df["operating_activity"], df["net_profit"])
ratios["capex_to_sales_pct"] = safe_divide(ratios["capex_cr"], df["sales"]) * 100
ratios["fcf_to_ebitda"] = safe_divide(ratios["free_cash_flow_cr"], ebitda)

# -----------------------------
# 4. Per Share KPIs
# -----------------------------
share_count = safe_divide(df["equity_capital"], df["face_value"])

ratios["earnings_per_share"] = df["eps"]
ratios["book_value_per_share"] = safe_divide(equity, share_count)
ratios["dividend_payout_ratio_pct"] = df["dividend_payout"]

# -----------------------------
# 5. Valuation KPIs
# -----------------------------
ratios["market_cap_crore"] = df["market_cap_crore"]
ratios["enterprise_value_crore"] = df["enterprise_value_crore"]
ratios["pe_ratio"] = df["pe_ratio"]
ratios["pb_ratio"] = df["pb_ratio"]
ratios["ev_ebitda"] = df["ev_ebitda"]
ratios["dividend_yield_pct"] = df["dividend_yield_pct"]
ratios["earnings_yield_pct"] = safe_divide(1, df["pe_ratio"]) * 100
ratios["fcf_yield_pct"] = safe_divide(ratios["free_cash_flow_cr"], df["market_cap_crore"]) * 100

ratios.head()



,company_id,year,fiscal_year,net_profit_margin_pct,operating_profit_margin_pct,pbt_margin_pct,ebit_margin_pct,expense_ratio_pct,return_on_equity_pct,return_on_capital_employed_pct,...,book_value_per_share,dividend_payout_ratio_pct,market_cap_crore,enterprise_value_crore,pe_ratio,pb_ratio,ev_ebitda,dividend_yield_pct,earnings_yield_pct,fcf_yield_pct
0,ABB,Dec 2012,2012.0,8.771930,12.220206,13.006655,11.070780,87.779794,22.411128,28.284389,...,308.095238,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABB,Mar 2014,2014.0,8.699473,11.731107,12.961336,10.764499,88.268893,25.126904,31.091371,...,375.238095,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABB,Mar 2015,2015.0,10.004369,13.630406,15.028397,12.975098,86.369594,24.439701,31.696905,...,446.190476,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABB,Mar 2016,2016.0,9.755164,13.963275,15.225708,13.427697,86.074981,21.338912,29.372385,...,569.047619,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABB,Mar 2017,2017.0,9.541853,13.709955,15.018946,13.158801,86.290045,19.971161,27.541456,...,660.476190,31.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
import os

# Create the directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save to CSV
ratios.to_csv("../data/processed/financial_ratios_generated.csv", index=False)

# Save to SQLite table
ratios.to_sql("financial_ratios_generated", conn, if_exists="replace", index=False)

print("Ratio Engine Version 1 completed!")
print("Rows:", len(ratios))
print("Columns:", len(ratios.columns))

Ratio Engine Version 1 completed!
Rows: 1079
Columns: 37


In [23]:
print("Shape:", ratios.shape)

print("Duplicate company-year rows:")
print(ratios.duplicated(["company_id", "year"]).sum())

print("Missing values:")
print(ratios.isna().sum().sort_values(ascending=False).head(15))

ratios.head()

Shape: (1079, 37)
Duplicate company-year rows:
0
Missing values:
fcf_yield_pct             545
market_cap_crore          539
pe_ratio                  539
pb_ratio                  539
dividend_yield_pct        539
earnings_yield_pct        539
ev_ebitda                 539
enterprise_value_crore    539
interest_coverage          43
book_value_per_share       36
fcf_to_ebitda              29
cfo_margin_pct             17
fcf_margin_pct             17
capex_to_sales_pct         17
cfo_to_pat                 17
dtype: int64


,company_id,year,fiscal_year,net_profit_margin_pct,operating_profit_margin_pct,pbt_margin_pct,ebit_margin_pct,expense_ratio_pct,return_on_equity_pct,return_on_capital_employed_pct,...,book_value_per_share,dividend_payout_ratio_pct,market_cap_crore,enterprise_value_crore,pe_ratio,pb_ratio,ev_ebitda,dividend_yield_pct,earnings_yield_pct,fcf_yield_pct
0,ABB,Dec 2012,2012.0,8.771930,12.220206,13.006655,11.070780,87.779794,22.411128,28.284389,...,308.095238,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABB,Mar 2014,2014.0,8.699473,11.731107,12.961336,10.764499,88.268893,25.126904,31.091371,...,375.238095,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABB,Mar 2015,2015.0,10.004369,13.630406,15.028397,12.975098,86.369594,24.439701,31.696905,...,446.190476,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABB,Mar 2016,2016.0,9.755164,13.963275,15.225708,13.427697,86.074981,21.338912,29.372385,...,569.047619,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABB,Mar 2017,2017.0,9.541853,13.709955,15.018946,13.158801,86.290045,19.971161,27.541456,...,660.476190,31.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
